In [ ]:
import numpy as np
import pandas as pd
import neurokit2 as nk

In [3]:
#Now we make the reshape function
#4 parameters, your dataframe, your window size or how many hz of rows should be in each row, purity for how much each row can tolerate label confounsion, and stride size for how much we should overlap
#By default this is every 10 seconds of data
def reshape_function(df, name = 'eda', window_size = 10, stride_size = 10, hertz=4, purity_threshold=0.8):
    window_row = window_size*hertz
    stride_row = stride_size*hertz
    #Get the subject names
    subjects = sorted(
        list(set(col.split("_")[0] for col in df.columns if "_" in col))
    )
    #Store the chunks of data while we're looping across the time span
    all_window = []
    for i in subjects:
        signal_col = f'{i}_{name}'
        label_col = f'{i}_Label'
        #Kill NAs in individual sequence
        individual_df = df[[signal_col,label_col]].dropna()
        total_rows = len(individual_df)
        #For each subject we take a chunk of their data by our determined window size then stack onto a list (in this case "all window")
        #Chunk is determined through standard range check then we check the purity threshold if we keep the chunk, based on birken's paper design
        for j in range(0,total_rows-window_row+1, stride_row):
            end = j + window_row
            signal_window = individual_df[signal_col].iloc[j:end].values
            label_window = individual_df[label_col].iloc[j:end].values

            # Determine the majority label and calculate if there exists a supermajority in the chunk to determine if kept
            value_counts = pd.Series(label_window).value_counts()
            majority_label = value_counts.index[0]
            majority_count = value_counts.iloc[0]

            purity = majority_count / window_row

            # Retain only windows meeting or exceeding the requested purity ratio
            if purity >= purity_threshold:
                row_data = [i, majority_label] + list(signal_window)
                all_window.append(row_data)

    feature_cols = [f"sample_{i}" for i in range(window_row)]
    columns = ["subject_id", "label"] + feature_cols

    return pd.DataFrame(all_window, columns=columns)

    





I also want visualization boxplots for each feature set similar to Zhu lili's breakdown

In [4]:
tonic = pd.read_csv("../Data/wesad_eda/tonic_signal.csv")
phasic = pd.read_csv("../Data/wesad_eda/phasic_signal.csv")

In [5]:
tonic_10 = reshape_function(tonic, name='Tonic')

In [6]:
tonic_10

,subject_id,label,sample_0,sample_1,sample_2,sample_3,sample_4,sample_5,sample_6,sample_7,...,sample_30,sample_31,sample_32,sample_33,sample_34,sample_35,sample_36,sample_37,sample_38,sample_39
0,S10,0.0,0.005656,0.005722,0.006034,0.006580,0.007352,0.008339,0.009532,0.010920,...,0.073866,0.076955,0.079991,0.082962,0.085860,0.088675,0.091396,0.094013,0.096517,0.098898
1,S10,0.0,0.101145,0.103248,0.105213,0.107044,0.108745,0.110321,0.111777,0.113117,...,0.122514,0.122450,0.122385,0.122323,0.122270,0.122229,0.122205,0.122203,0.122227,0.122283
2,S10,0.0,0.122374,0.122505,0.122674,0.122880,0.123121,0.123395,0.123700,0.124034,...,0.135711,0.136213,0.136699,0.137168,0.137618,0.138047,0.138454,0.138836,0.139192,0.139519
3,S10,0.0,0.139818,0.140084,0.140320,0.140527,0.140706,0.140857,0.140983,0.141084,...,0.139251,0.139106,0.138965,0.138829,0.138700,0.138578,0.138465,0.138362,0.138270,0.138190
4,S10,0.0,0.138123,0.138071,0.138033,0.138008,0.137995,0.137995,0.138006,0.138027,...,0.139970,0.140053,0.140131,0.140202,0.140266,0.140323,0.140372,0.140412,0.140443,0.140464
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7675,S9,0.0,0.385685,0.385735,0.385782,0.385825,0.385865,0.385901,0.385935,0.385966,...,0.386156,0.386154,0.386153,0.386153,0.386152,0.386153,0.386154,0.386156,0.386159,0.386164
7676,S9,0.0,0.386169,0.386176,0.386184,0.386191,0.386197,0.386201,0.386203,0.386200,...,0.383187,0.382843,0.382474,0.382079,0.381658,0.381209,0.380732,0.380226,0.379690,0.379123
7677,S9,0.0,0.378525,0.377894,0.377232,0.376539,0.375817,0.375065,0.374286,0.373480,...,0.349597,0.348418,0.347235,0.346048,0.344859,0.343669,0.342478,0.341288,0.340099,0.338913
7678,S9,0.0,0.337730,0.336550,0.335376,0.334207,0.333043,0.331887,0.330737,0.329596,...,0.306907,0.306137,0.305390,0.304668,0.303969,0.303296,0.302649,0.302027,0.301433,0.300866


In [7]:
sample_cols = [col for col in tonic_10.columns if col.startswith('sample_')]
tonic_descriptives = pd.DataFrame()
#Bring Labels into descriptives to the databas
tonic_descriptives['label'] = tonic_10['label']
#Bring subject ID over to descriptives
if 'subject_id' in tonic_10.columns:
    tonic_descriptives['subject_id'] = tonic_10['subject_id']
#Descriptives
tonic_descriptives['tonic_mean'] = tonic_10[sample_cols].mean(axis=1)
tonic_descriptives['tonic_max']  = tonic_10[sample_cols].max(axis=1)

tonic_descriptives

,label,subject_id,tonic_mean,tonic_max
0,0.0,S10,0.044755,0.098898
1,0.0,S10,0.118371,0.122666
2,0.0,S10,0.130307,0.139519
3,0.0,S10,0.140129,0.141265
4,0.0,S10,0.139057,0.140464
...,...,...,...,...
7675,0.0,S9,0.386073,0.386164
7676,0.0,S9,0.384511,0.386203
7677,0.0,S9,0.360526,0.378525
7678,0.0,S9,0.317308,0.337730


In [8]:
phasic_10 = reshape_function(phasic, name='Phasic')

In [ ]:
sample_cols = [col for col in phasic_10.columns if col.startswith('sample_')]
phasic_descriptives = pd.DataFrame()
#Bring Labels into descriptives to the database
phasic_descriptives['label'] = phasic_10['label']
#Bring subject ID over to descriptives
if 'subject_id' in phasic_10.columns:
    phasic_descriptives['subject_id'] = phasic_10['subject_id']
#Descriptives
phasic_descriptives['phasic_mean'] = phasic_10[sample_cols].mean(axis=1)
phasic_descriptives['peaks_count'] = (phasic_10[sample_cols] > 0.01).sum(axis=1)
phasic_descriptives

,label,subject_id,phasic_mean,peaks_count
0,0.0,S10,4.818524e-02,17
1,0.0,S10,1.144845e-04,0
2,0.0,S10,2.629398e-03,3
3,0.0,S10,8.828982e-08,0
4,0.0,S10,7.839005e-08,0
...,...,...,...,...
7675,0.0,S9,3.115321e-07,0
7676,0.0,S9,3.058195e-05,0
7677,0.0,S9,1.752075e-02,14
7678,0.0,S9,2.946647e-02,8


In [ ]:
row_example = phasic_10.iloc[0:1,2:]
signals, info = nk.eda_peaks(np.array(row_example).flatten(), sampling_rate=4)
# Calculate rise times in seconds
rise_times_sec = (info["SCR_Peaks"] - info["SCR_Onsets"]) / 4
np.nanstd(rise_times_sec)


0.2915475947422651

In [ ]:
def rise_time_std(row, sampling_rate = 4):
    try:
        #Uses neurokits peak detection to get the onsets then convert it to a time scale and calculate the standard deviation
        signals, info = nk.eda_peaks(row, sampling_rate=sampling_rate)
        rise_times = (info["SCR_Peaks"] - info["SCR_Onsets"]) / sampling_rate
        return np.nanstd(rise_times)
    #There is a lot of rows with no peaks at all, need to decide how to handle the NAs where the std depreciates thereof
    except Exception:
        return np.nan
#Create a new column after apply row wise the above function
phasic_descriptives['phasic_std_rise_time'] = phasic_10[sample_cols].apply(
    lambda row: rise_time_std(row), axis=1
)

phasic_descriptives

/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/home/sj/.local/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/home/sj/.local/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice

,label,subject_id,phasic_mean,peaks_count,phasic_std_rise_time
0,0.0,S10,4.818524e-02,17,0.291548
1,0.0,S10,1.144845e-04,0,0.000000
2,0.0,S10,2.629398e-03,3,0.375000
3,0.0,S10,8.828982e-08,0,1.000000
4,0.0,S10,7.839005e-08,0,0.000000
...,...,...,...,...,...
7675,0.0,S9,3.115321e-07,0,NaN
7676,0.0,S9,3.058195e-05,0,0.000000
7677,0.0,S9,1.752075e-02,14,0.515079
7678,0.0,S9,2.946647e-02,8,1.532065


In [34]:
feature_dataframe = pd.concat(
    [phasic_descriptives, tonic_descriptives.drop(columns=["subject_id", "label"])],
    axis=1,
)
feature_dataframe

,label,subject_id,phasic_mean,peaks_count,phasic_std_rise_time,tonic_mean,tonic_max
0,0.0,S10,4.818524e-02,17,0.291548,0.044755,0.098898
1,0.0,S10,1.144845e-04,0,0.000000,0.118371,0.122666
2,0.0,S10,2.629398e-03,3,0.375000,0.130307,0.139519
3,0.0,S10,8.828982e-08,0,1.000000,0.140129,0.141265
4,0.0,S10,7.839005e-08,0,0.000000,0.139057,0.140464
...,...,...,...,...,...,...,...
7675,0.0,S9,3.115321e-07,0,NaN,0.386073,0.386164
7676,0.0,S9,3.058195e-05,0,0.000000,0.384511,0.386203
7677,0.0,S9,1.752075e-02,14,0.515079,0.360526,0.378525
7678,0.0,S9,2.946647e-02,8,1.532065,0.317308,0.337730


In [35]:
feature_dataframe.to_csv("../Data/wesad_eda/feature_dataframe.csv", index=False)
